# Generating Observations

### Loading Python Modules:

In [ ]:
import numpy as np
import random as rd
import pandas as pd
from class_state_vec import state_vector
from class_obs import obs_da

### Loading Lorenz63 run data from running "gen_nature_run.ipynb" and defining "x_nature" run:

In [ ]:
#infile      = 'lorenz63_run-001.pkl'    # Qiwen's run
infile      = 'lorenz63_run.pkl'    # Diego's run

sv          = state_vector()
sv          = sv.load(infile)
sv.setFigurePath('Python_Figures/')
x_nature    = sv.getTrajectory()  # The trajectory obtained from "gen_nature_run.ipynb" is stored in x_nature
x_nature    = x_nature            # Qiwen's run
xdim, maxit = np.shape(x_nature)
std_nature  = np.zeros(xdim)
print('xdim =', xdim)    #row
print('maxit = ', maxit) #column
print('x_nature shape:', x_nature.shape)

#
# Plot the nature run
#
sv.plotTrajectory(x_nature, 'Lorenz-63 State $x$ Natural Run', 'nature_run.png')

### Compute the standard deviation of each state:

In [ ]:
for i in range(xdim):
    std_nature[i] = np.std(x_nature[i,:])

print('std_nature =', std_nature)

### Generating Synthetic Observations:

In [ ]:
outfile = 'y_obs.pkl'
obs     = obs_da(name = 'observe_full_state')
yo      = np.zeros_like(x_nature) # return the same shape as x_nature with zeros

np.random.seed(6) # set the seed for reproducibility

for i in range(xdim):
    yo[i,:] = x_nature[i,:] + np.random.normal(0, np.sqrt(1.5), maxit)  # add noise to the true state

obs.setVal(yo) # set the value of yo to obs class

#-----------------------------------------------------------------------
# Computing observation position:
#-----------------------------------------------------------------------

pos = np.zeros_like(yo)
for i in range(maxit):
    pos[:,i] = list(range(0,3))
    
obs.setPos(pos)

print('shape yo =', np.shape(yo))
print('first few obs =', yo[:, :5])

#-----------------------------------------------------------------------
# Store the true and noisy observations:
#-----------------------------------------------------------------------

obs.save(outfile)

### Plotting PDF of the Synthetic Observations:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

PathFigures = 'figures/'

# Create figure and axis
fig, ax = plt.subplots(figsize=(10, 5))

# Define colors and labels for each observation channel
colors = ['green', 'red', 'blue']
labels = ['Observation X', 'Observation Y', 'Observation Z']

# Plot PDFs (histograms normalized to show probability density)
for i in range(3):
    ax.hist(yo[i, :], bins=100, density=True, alpha=0.6, 
            color=colors[i], label=labels[i], edgecolor='black')

# Add labels and title
ax.set_xlabel('Observation Value', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title('PDFs of Observations for Each Model State', fontsize=14)

# Add legend and grid for better readability
ax.legend()
ax.grid(True)

plt.show()
fig.savefig(PathFigures + 'obs_pdf.png', dpi=300)
plt.close()


### Loading Observations:

In [ ]:
#-----------------------------------------------------------------------
# Read the Lorenz-96 observations
#-----------------------------------------------------------------------
infile = 'y_obs.pkl'
obs = obs_da()
obs = obs.load(infile)

#-----------------------------------------------------------------------
# Get the Observed States
#-----------------------------------------------------------------------
y_obs = obs.getVal()
y_pos = obs.getPos()
y_err = obs.getErr()

print('y_obs =', y_obs)
print('y_err =', y_err)
print('y_pos =', y_pos)